In [1]:
import pandas as pd
import numpy as np

In [8]:
df=pd.read_csv(r"C:\Users\samru\OneDrive\Desktop\SIH_2025\AI-Pollution-Forecast-and-Policy-Dashboard\ML\Raw\CCR_DATA_2023_25\Raw_data_1Day_2024_site_104_Burari_Crossing_Delhi_IMD_1Day.csv")

In [9]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),NH3 (µg/m³),SO2 (µg/m³),CO (mg/m³),Ozone (µg/m³),...,MP-Xylene (µg/m³),AT (°C),RH (%),WS (m/s),WD (deg),RF (mm),TOT-RF (mm),SR (W/mt2),BP (mmHg),VWS (m/s)
0,2024-01-01,143.29,278.27,14.57,6.17,20.74,NaN,NaN,2.29,17.47,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
1,2024-01-02,146.48,258.60,10.50,4.49,14.98,NaN,NaN,2.50,19.27,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
2,2024-01-03,154.35,261.26,14.20,7.07,21.24,NaN,NaN,2.10,19.16,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
3,2024-01-04,223.31,377.21,30.07,14.06,44.13,NaN,NaN,1.89,27.56,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
4,2024-01-05,128.92,255.26,27.28,12.71,39.99,NaN,NaN,2.06,24.94,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
361,2024-12-27,123.01,153.32,24.94,13.82,27.62,NaN,NaN,0.41,18.48,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
362,2024-12-28,80.48,105.64,28.05,13.25,29.85,NaN,NaN,0.33,10.97,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
363,2024-12-29,74.46,119.38,25.33,11.59,26.75,NaN,NaN,0.08,22.35,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN
364,2024-12-30,81.31,127.57,11.62,12.87,16.30,NaN,NaN,0.45,28.42,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN


In [10]:
# ---------- 2. Remove duplicate rows and columns ----------
df = df.drop_duplicates().reset_index(drop=True)
df = df.loc[:, ~df.T.duplicated()]
print("Shape after removing duplicates:", df.shape)

Shape after removing duplicates: (366, 10)


In [11]:
# ---------- 3. Handle missing values (drop >70% NaN, impute median otherwise) ----------
nan_thresh = 0.7

# Drop columns with >70% missing
cols_to_drop = df.columns[df.isnull().mean() > nan_thresh]
df = df.drop(columns=cols_to_drop)
print(f"Dropped columns (>{int(nan_thresh*100)}% NaN): {cols_to_drop.tolist()}")

# Drop rows with >70% missing
rows_to_drop = df.index[df.isnull().mean(axis=1) > nan_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
print(f"Dropped rows (>{int(nan_thresh*100)}% NaN):", len(rows_to_drop))

# Impute remaining missing values
num_cols = df.select_dtypes(include=[np.number]).columns
cat_cols = df.select_dtypes(exclude=[np.number]).columns

for col in num_cols:
    median_val = df[col].median()
    df[col] = df[col].fillna(median_val)

for col in cat_cols:
    if df[col].isnull().any():
        mode_val = df[col].mode(dropna=True)
        if not mode_val.empty:
            df[col] = df[col].fillna(mode_val[0])

print("Missing values after imputation:\n", df.isnull().sum())

Dropped columns (>70% NaN): ['NH3 (µg/m³)']
Dropped rows (>70% NaN): 3
Missing values after imputation:
 Timestamp        0
PM2.5 (µg/m³)    0
PM10 (µg/m³)     0
NO (µg/m³)       0
NO2 (µg/m³)      0
NOx (ppb)        0
CO (mg/m³)       0
Ozone (µg/m³)    0
TOT-RF (mm)      0
dtype: int64


In [12]:
# ---------- 4. Handle outliers using IQR (with 70% rule) ----------

def get_outlier_mask(series):
    q1 = series.quantile(0.25)
    q3 = series.quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    return (series < lower) | (series > upper)

# Handle columns: drop if >70% outliers, else replace with median
outlier_thresh = 0.7
cols_to_drop = []
for col in num_cols:
    outlier_mask = get_outlier_mask(df[col])
    outlier_fraction = outlier_mask.mean()
    if outlier_fraction > outlier_thresh:
        cols_to_drop.append(col)
    else:
        median_val = df[col].median()
        df.loc[outlier_mask, col] = median_val
if cols_to_drop:
    df = df.drop(columns=cols_to_drop)
    print(f"Dropped numeric columns (>{int(outlier_thresh*100)}% outliers): {cols_to_drop}")
    
# Handle rows: drop if >70% numeric columns are outliers in a given row
outlier_matrix = df[num_cols].apply(get_outlier_mask)
row_outlier_fraction = outlier_matrix.mean(axis=1)
rows_to_drop = df.index[row_outlier_fraction > outlier_thresh]
df = df.drop(index=rows_to_drop).reset_index(drop=True)
if len(rows_to_drop) > 0:
    print(f"Dropped rows (>{int(outlier_thresh*100)}% outliers): {len(rows_to_drop)}")


In [13]:
# ---------- 6. Final check ----------
print("Final shape:", df.shape)
print(df.head())

Final shape: (363, 9)
    Timestamp  PM2.5 (µg/m³)  PM10 (µg/m³)  NO (µg/m³)  NO2 (µg/m³)  \
0  2024-01-01         143.29        278.27       14.57         6.17   
1  2024-01-02         146.48        258.60       10.50         4.49   
2  2024-01-03         154.35        261.26       14.20         7.07   
3  2024-01-04         223.31        377.21       30.07        14.06   
4  2024-01-05         128.92        255.26       27.28        12.71   

   NOx (ppb)  CO (mg/m³)  Ozone (µg/m³)  TOT-RF (mm)  
0      20.74        2.29          17.47          0.0  
1      14.98        2.50          19.27          0.0  
2      21.24        2.10          19.16          0.0  
3      44.13        1.89          27.56          0.0  
4      39.99        2.06          24.94          0.0  


In [14]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
numerics = df.select_dtypes(include=[np.number]).columns
df[numerics] = scaler.fit_transform(df[numerics])

In [15]:
df

,Timestamp,PM2.5 (µg/m³),PM10 (µg/m³),NO (µg/m³),NO2 (µg/m³),NOx (ppb),CO (mg/m³),Ozone (µg/m³),TOT-RF (mm)
0,2024-01-01,0.729002,0.627006,0.116098,-1.044737,-0.316143,1.633635,-1.260805,0.0
1,2024-01-02,0.780081,0.445162,-0.352132,-1.368675,-0.982826,1.956399,-1.115871,0.0
2,2024-01-03,0.906099,0.469753,0.073532,-0.871198,-0.258271,1.341611,-1.124728,0.0
3,2024-01-04,2.010313,1.541683,1.899284,0.476617,2.391101,1.018847,-0.448370,0.0
4,2024-01-05,0.498904,0.414285,1.578310,0.216310,1.911923,1.280132,-0.659330,0.0
...,...,...,...,...,...,...,...,...,...
358,2024-12-27,0.404270,-0.528126,1.309107,0.430340,0.480174,-1.255871,-1.179481,0.0
359,2024-12-28,-0.276736,-0.968916,1.666894,0.320433,0.738282,-1.378828,-1.784178,0.0
360,2024-12-29,-0.373131,-0.841893,1.353974,0.000351,0.379477,-1.763071,-0.867873,0.0
361,2024-12-30,-0.263446,-0.766178,-0.223282,0.247161,-0.830045,-1.194392,-0.379124,0.0


In [16]:
df.to_excel('Burari2024.xlsx', index=False)